# 1. Fitting the null GLMM

`-` 목적: 공변량 효과와 개인 유전적 차이를 덜어낸 잔차를 산출

$$ y = X_i\alpha + b_i + \epsilon_i$$

`1` 개별 변이 효과가 없는 귀무모형 적합

> We used the average information restricted maximum-likelihood algorithm to fit the null GLMM (that is, $H_0: \boldsymbol{\beta}_j = \mathbf{0}$) as in SAIGE step 1.

`2` 귀무가설($H_0: \boldsymbol{\beta}_j = \mathbf{0}$) 하에서 SAIGE의 AI-REML 알고리즘을 돌려 다음 값들을 추정함

- $\hat {\alpha}$: 나이, 성별, 유전적 주성분(PC) 등 **공변량의 고정 효과 계수**

- $\hat{\sigma}_g^2, \hat{\sigma}_e^2$: 가계 혈연 및 전체 배경 유전체가 차지하는 배경 유전 분산과 잔차 환경 분산 

`3` 즉, 원래 표현형($y_i$)에서 **"나이·성별 때문에 생긴 차이($X_i{\alpha}$)"** 와 **"집안 내력(혈연) 때문에 생긴 차이($\beta_j$)"** 를 수학적으로 완전히 도려내고 남은 **순수 잔차($\tilde{\mathbf{y}}$)** 를 손에 쥐는 것이 목적

`4` "AI-REML": `Average Information Restricted Maximum Likelihood`

- 대충 아래와 같은 사이클....

- $\sigma_g^2$ 초깃값 가정 $\rightarrow$ $\boldsymbol{\alpha}$ 추정 $\rightarrow$ 잔차 계산 $\rightarrow$ $\sigma_g^2$ 업데이트 $\rightarrow$ 다시 $\boldsymbol{\alpha}$ 재추정...

> **Gilmour et al. (1995)**: *Average information REML: an efficient algorithm for variance parameter estimation in linear mixed models* (AI-REML의 수학적 유도 원논문)

<details>

<summary>기호 의미</summary>

`-` 개별 피험자 기준

- $g(\mu_{i})$: $\mu$ or $\text{logit}$

- $X_{i}$: $i$번째 사람의 나이, 성별 등

- $b_i$: $i$번째 개인의 유전적 형질로 인해 발생한 **random effect**

- $\sigma_g$: Polygenic Variance component

- $\epsilon$: error term

- $K$: Genentic Relatedenss Matrix, GRM $\to$ 피험자들끼리 유전적으로 얼마나 닮았는지 나타냄.

    - $G_{im} \in \{0, 1, 2\}$: $i$번째 개인의 $m$번째 SNP 대립유전자 개수, $G_{im} \sim \text{Binomial}(2, p_m)$
 
    - $p_m$: 코호트 전체에서 $m$번째 SNP의 대립유전자 빈도(MAF, Minor Allele Frequency)

$$\mathbf{K} = \frac{1}{M} \mathbf{Z}\mathbf{Z}^T$$

$$\mathbf{Z} = \begin{bmatrix} \frac{G_{11} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{12} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{1M} - 2p_M}{\sqrt{2p_M(1-p_M)}} \\ \frac{G_{21} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{22} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{2M} - 2p_M}{\sqrt{2p_M(1-p_M)}} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{G_{n1} - 2p_1}{\sqrt{2p_1(1-p_1)}} & \frac{G_{n2} - 2p_2}{\sqrt{2p_2(1-p_2)}} & \cdots & \frac{G_{nM} - 2p_M}{\sqrt{2p_M(1-p_M)}} \end{bmatrix}_{n \times M}$$

$$\begin{bmatrix} b_1 \\ b_2 \\ b_3 \\ \vdots \\ b_n \end{bmatrix} \sim \text{MVN}\left( \begin{bmatrix} 0 \\ 0 \\ 0 \\ \vdots \\ 0 \end{bmatrix}, \;\; \sigma_g^2 \begin{bmatrix} K_{11} & K_{12} & K_{13} & \cdots & K_{1n} \\ K_{21} & K_{22} & K_{23} & \cdots & K_{2n} \\ K_{31} & K_{32} & K_{33} & \cdots & K_{3n} \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ K_{n1} & K_{n2} & K_{n3} & \cdots & K_{nn} \end{bmatrix} \right)$$

`5` 그래서 최종 $\hat y$ 산출

$$\hat{y}_i = \hat{\mu}_i = X_i\hat{\boldsymbol{\alpha}} + \hat{b}_i$$

</detials>

---

# 2. estimate of gene-level (region-level) heritability

`1` step1에서 공변량($X_i \hat{\alpha}$) 효과와 개인의 유전적 형질 차이($\hat b_i$)를 덜어난 $\tilde{\mathbf{y}}$ 추출

$$\tilde{\mathbf{y}} = \mathbf{G}_j \boldsymbol{\beta}_j + \boldsymbol{\epsilon}$$

- $\mathbf{G}_j$: Genotype matrix

- $\boldsymbol{\beta}_j \sim \text{MVN}(\mathbf{0}, \tau \boldsymbol{\Sigma})$: 희귀변이들의 random effect

    - SKAT과 똑같이 $\tau$만 추정

- $\boldsymbol{\epsilon} \sim \text{MVN}(\mathbf{0}, \psi \mathbf{I}_n)$: noise

`2` To mitigate the computational demands, we adopt FaST-LMM.

>"Given $n \gg k$, FaST-LMM reduces the computation cost from $O(n^3)$ of conventional mixed-model algorithm to $O(nk^2)$, where $n$ is the number of samples in the dataset and $k$ is the number of genetic variants in a single group (Methods)."